In [ ]:
# ============================================================
# PROJECT ANALYTICS PRO — Excel Sheet Merger
# Google Colab | Divishath | Feb 2026
# ============================================================
# Sheets expected in your workbook:
#   Sheet1  → Project_ID, Phase, Date, Planned_Duration,
#              Planned_Cost, Planned_Delivrable
#   Sheet2  → Project_ID, Project_Type
#   Sheet3  → Project_ID, Phase, Actual_Cost
#   Sheet4  → Project_ID, Phase, Actual_Duration
#   Sheet5  → Project_ID, Phase, Actual_Deliverables
#   Sheet6  → Project_ID, Country
#   Sheet7  → Country, Region, Type
# ============================================================

# ── STEP 0: Install & Import ─────────────────────────────────
!pip install openpyxl --quiet

import pandas as pd
from google.colab import files

# ── STEP 1: Upload your Excel workbook ───────────────────────
print("📂 Please upload your Excel file (.xlsx) ...")
uploaded = files.upload()

# Auto-detect the uploaded filename
filename = list(uploaded.keys())[0]
print(f"\n✅ File uploaded: {filename}")

# ── STEP 2: Load all sheets ───────────────────────────────────
print("\n📊 Loading sheets...")

sheet1 = pd.read_excel(filename, sheet_name="Sheet1")   # Planned data
sheet2 = pd.read_excel(filename, sheet_name="Sheet2")   # Project Type
sheet3 = pd.read_excel(filename, sheet_name="Sheet3")   # Actual Cost
sheet4 = pd.read_excel(filename, sheet_name="Sheet4")   # Actual Duration
sheet5 = pd.read_excel(filename, sheet_name="Sheet5")   # Actual Deliverables
sheet6 = pd.read_excel(filename, sheet_name="Sheet6")   # Country
sheet7 = pd.read_excel(filename, sheet_name="Sheet7")   # Region & Type

print(f"  Sheet1 (Planned)       → {sheet1.shape[0]} rows")
print(f"  Sheet2 (Project Type)  → {sheet2.shape[0]} rows")
print(f"  Sheet3 (Actual Cost)   → {sheet3.shape[0]} rows")
print(f"  Sheet4 (Actual Dur.)   → {sheet4.shape[0]} rows")
print(f"  Sheet5 (Deliverables)  → {sheet5.shape[0]} rows")
print(f"  Sheet6 (Country)       → {sheet6.shape[0]} rows")
print(f"  Sheet7 (Region/Type)   → {sheet7.shape[0]} rows")

# ── STEP 3: Strip whitespace from column names ────────────────
for df in [sheet1, sheet2, sheet3, sheet4, sheet5, sheet6, sheet7]:
    df.columns = df.columns.str.strip()

# ── STEP 4: Standardise Project_ID column name ───────────────
# Sheet2 might use different casing — force rename safely
sheet2.rename(columns={"Project_ID": "Project_ID"}, inplace=True)
sheet6.rename(columns={"Project_ID": "Project_ID"}, inplace=True)

# ── STEP 5: Start merging — base = Sheet1 ────────────────────
print("\n🔗 Merging tables...")

# 5a. Merge Project_Type from Sheet2 (on Project_ID)
fact = sheet1.merge(
    sheet2[["Project_ID", "Project_Type"]],
    on="Project_ID",
    how="left"
)
print(f"  After + Project_Type   → {fact.shape[0]} rows, {fact.shape[1]} cols")

# 5b. Merge Actual_Cost from Sheet3 (on Project_ID + Phase)
fact = fact.merge(
    sheet3[["Project_ID", "Phase", "Actual_Cost"]],
    on=["Project_ID", "Phase"],
    how="left"
)
print(f"  After + Actual_Cost    → {fact.shape[0]} rows, {fact.shape[1]} cols")

# 5c. Merge Actual_Duration from Sheet4 (on Project_ID + Phase)
fact = fact.merge(
    sheet4[["Project_ID", "Phase", "Actual_Duration"]],
    on=["Project_ID", "Phase"],
    how="left"
)
print(f"  After + Actual_Duration→ {fact.shape[0]} rows, {fact.shape[1]} cols")

# 5d. Merge Actual_Deliverables from Sheet5 (on Project_ID + Phase)
fact = fact.merge(
    sheet5[["Project_ID", "Phase", "Actual_Deliverables"]],
    on=["Project_ID", "Phase"],
    how="left"
)
print(f"  After + Deliverables   → {fact.shape[0]} rows, {fact.shape[1]} cols")

# 5e. Merge Country from Sheet6 (on Project_ID)
fact = fact.merge(
    sheet6[["Project_ID", "Country"]],
    on="Project_ID",
    how="left"
)
print(f"  After + Country        → {fact.shape[0]} rows, {fact.shape[1]} cols")

# 5f. Merge Region & Type from Sheet7 (on Country)
fact = fact.merge(
    sheet7[["Country", "Region", "Type"]],
    on="Country",
    how="left"
)
print(f"  After + Region/Type    → {fact.shape[0]} rows, {fact.shape[1]} cols")

# ── STEP 6: Reorder columns cleanly ──────────────────────────
final_columns = [
    "Project_ID",
    "Project_Type",
    "Phase",
    "Date",
    "Country",
    "Region",
    "Type",
    "Planned_Duration",
    "Planned_Cost",
    "Planned_Delivrable",
    "Actual_Cost",
    "Actual_Duration",
    "Actual_Deliverables"
]

# Only keep columns that exist (safe guard)
final_columns = [c for c in final_columns if c in fact.columns]
fact = fact[final_columns]

# ── STEP 7: Add Derived / Calculated Columns ─────────────────
fact["Cost_Variance"]      = fact["Actual_Cost"]      - fact["Planned_Cost"]
fact["Duration_Variance"]  = fact["Actual_Duration"]  - fact["Planned_Duration"]
fact["Deliverables_Variance"] = fact["Actual_Deliverables"] - fact["Planned_Delivrable"]

# Cost Variance % (avoid divide by zero)
fact["Cost_Variance_Pct"] = (
    fact["Cost_Variance"] / fact["Planned_Cost"].replace(0, pd.NA) * 100
).round(2)

# Project Health Status
def health_status(row):
    try:
        if row["Actual_Cost"] <= row["Planned_Cost"] and \
           row["Actual_Duration"] <= row["Planned_Duration"]:
            return "On Track"
        elif row["Actual_Cost"] > row["Planned_Cost"] * 1.2:
            return "Over Budget"
        else:
            return "At Risk"
    except:
        return "Unknown"

fact["Health_Status"] = fact.apply(health_status, axis=1)

# ── STEP 8: Quality Check ─────────────────────────────────────
print("\n🔍 Data Quality Check:")
print(f"  Total rows             : {fact.shape[0]}")
print(f"  Total columns          : {fact.shape[1]}")
print(f"\n  Null values per column:")
print(fact.isnull().sum().to_string())
print(f"\n  Health Status breakdown:")
print(fact["Health_Status"].value_counts().to_string())

# ── STEP 9: Preview the merged table ─────────────────────────
print("\n📋 Preview (first 5 rows):")
print(fact.head().to_string())

# ── STEP 10: Export to CSV and download ──────────────────────
output_filename = "Fact_Projects_Merged.csv"
fact.to_csv(output_filename, index=False)
print(f"\n✅ CSV saved as: {output_filename}")

# Auto-download the CSV to your computer
files.download(output_filename)
print("⬇️  Download started!")

# ── STEP 11: Also export as Excel (bonus) ────────────────────
output_excel = "Fact_Projects_Merged.xlsx"
with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    fact.to_excel(writer, sheet_name="Fact_Projects", index=False)
print(f"✅ Excel also saved as: {output_excel}")
files.download(output_excel)
print("⬇️  Excel download started!")


📂 Please upload your Excel file (.xlsx) ...


Saving Tableau_project_1.xlsx to Tableau_project_1 (2).xlsx

✅ File uploaded: Tableau_project_1 (2).xlsx

📊 Loading sheets...
  Sheet1 (Planned)       → 520 rows
  Sheet2 (Project Type)  → 104 rows
  Sheet3 (Actual Cost)   → 520 rows
  Sheet4 (Actual Dur.)   → 520 rows
  Sheet5 (Deliverables)  → 520 rows
  Sheet6 (Country)       → 104 rows
  Sheet7 (Region/Type)   → 52 rows

🔗 Merging tables...
  After + Project_Type   → 520 rows, 7 cols
  After + Actual_Cost    → 520 rows, 8 cols
  After + Actual_Duration→ 520 rows, 9 cols
  After + Deliverables   → 520 rows, 10 cols
  After + Country        → 520 rows, 11 cols
  After + Region/Type    → 520 rows, 13 cols

🔍 Data Quality Check:
  Total rows             : 520
  Total columns          : 18

  Null values per column:
Project_ID               0
Project_Type             0
Phase                    0
Date                     0
Country                  0
Region                   0
Type                     0
Planned_Duration         0
Planned_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Download started!
✅ Excel also saved as: Fact_Projects_Merged.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Excel download started!
